## Fake Review Detection

In [1]:
import pandas as pd 

In [2]:
df=pd.read_csv("FakeReview.csv")
df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40432 entries, 0 to 40431
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   category  40432 non-null  object 
 1   rating    40432 non-null  float64
 2   label     40432 non-null  object 
 3   text_     40432 non-null  object 
dtypes: float64(1), object(3)
memory usage: 1.2+ MB


In [4]:
df.shape

(40432, 4)

In [5]:
df.isnull().sum() #Indicates No null Value

category    0
rating      0
label       0
text_       0
dtype: int64

In [6]:
df['rating'].value_counts()

rating
5.0    24559
4.0     7965
3.0     3786
1.0     2155
2.0     1967
Name: count, dtype: int64

In [7]:
print(df['label'].value_counts())

label
CG    20216
OR    20216
Name: count, dtype: int64


In [8]:
df[df.duplicated()]# Contains Duplicate Values Drop the Duplicates


,category,rating,label,text_
6025,Sports_and_Outdoors_5,5.0,CG,"This is a really good starter kit, with lots o..."
6708,Sports_and_Outdoors_5,5.0,CG,"Great, no complaints. Comfortable, phone fits ..."
12548,Movies_and_TV_5,5.0,CG,One of the best movies of the year. Not for e...
19802,Pet_Supplies_5,5.0,CG,My dog loves these and it has kept her occupie...
20242,Pet_Supplies_5,5.0,CG,My dog loves it and it has kept her occupied f...
22305,Pet_Supplies_5,5.0,OR,Got these to give to my 8 mth old chihuahua wh...
26444,Kindle_Store_5,5.0,CG,I received this story as an ARC in exchange fo...
27233,Kindle_Store_5,5.0,CG,This is the first book in a series by the auth...
29203,Books_5,5.0,CG,I really enjoyed this book. The characters wer...
33602,Toys_and_Games_5,5.0,CG,I got this for my son for Christmas. He loved...


In [9]:
df.drop_duplicates(inplace=True)

In [10]:
df.shape

(40420, 4)

In [11]:
print(df.duplicated().sum()) # Duplicates are Removed

0


In [12]:
df['label'].value_counts() #Indicates This balanced Dataset

label
OR    20215
CG    20205
Name: count, dtype: int64

#### Encode the Labels

In [13]:
df['label']=df['label'].map({'CG':0,'OR':1}) # CG -> computer Generated,
#OR original Review
df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,0,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,0,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,0,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,0,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,0,Very nice set. Good quality. We have had the s...


#### Text Cleaning 

In [14]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

In [15]:
df['text_'] = df['text_'].apply(clean_text)

In [16]:
df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,0,love this well made sturdy and very comfortab...
1,Home_and_Kitchen_5,5.0,0,love it a great upgrade from the original ive...
2,Home_and_Kitchen_5,5.0,0,this pillow saved my back i love the look and ...
3,Home_and_Kitchen_5,1.0,0,missing information on how to use it but it is...
4,Home_and_Kitchen_5,5.0,0,very nice set good quality we have had the set...


In [17]:
X=df['text_']
y=df['label']

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.3, random_state=42)

#### Convert the text To number using TF IDF


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(stop_words='english')
X_train_vector=vectorizer.fit_transform(X_train)
X_test_vector=vectorizer.transform(X_test)


#### HyperParameter Tuning 

#### Train the model 

In [21]:
from sklearn.ensemble import RandomForestClassifier
model=RandomForestClassifier(criterion='gini',random_state=42)
model.fit(X_train_vector,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


<h5>Grid SearchCV</h5>

In [22]:
from sklearn.model_selection import GridSearchCV


In [23]:
best_param=GridSearchCV(RandomForestClassifier(),
                       {
                           'n_estimators':[100,200],
                           'max_depth':[10,20]
                           
                       }, cv=5,scoring='accuracy'
                       )

In [24]:
best_param.fit(X_train_vector,y_train)

,estimator,RandomForestClassifier()
,param_grid,"{'max_depth': [10, 20], 'n_estimators': [100, 200]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [25]:
best_param.best_params_



{'max_depth': 20, 'n_estimators': 200}

In [26]:
best_param.best_score_

np.float64(0.8032090450363543)

In [27]:
#best_model=best_param.best_estimator_ 

<h5>takes Best model in best_parameter among 
Best parameters
Best score
Best model </h5>

In [29]:
y_pred =model.predict(X_test_vector)

In [30]:
model.score(X_train_vector,y_train)

1.0

#### Evaluation Matrix

In [37]:
from sklearn.metrics import classification_report
report=classification_report(y_test,y_pred)

In [40]:
print(report)

              precision    recall  f1-score   support

           0       0.81      0.89      0.85      6094
           1       0.88      0.79      0.83      6032

    accuracy                           0.84     12126
   macro avg       0.85      0.84      0.84     12126
weighted avg       0.85      0.84      0.84     12126



In [53]:
import pickle 

In [54]:
# save the model
with open("RFModel.pkl","wb") as file:
    pickle.dump(best_model,file)

In [55]:
# Save vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)